In [1]:
import os
import boto3
from sagemaker import get_execution_role
import time
from pprint import pprint
import shutil

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Constants

In [2]:
# name
str_function_name = 'genxii-ad-select-best-model'

### 1. Create container

### Create ```Dockerfile```

In [3]:
%%writefile Dockerfile

FROM public.ecr.aws/lambda/python:3.8

# update pip
RUN pip install --upgrade pip

# install dependencies from project folder
COPY requirements.txt  .
RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"

# copy function code
COPY lambda_function.py ${LAMBDA_TASK_ROOT}

# Set the CMD to your handler (could also be done as a parameter override outside of the Dockerfile)
CMD ["lambda_function.lambda_handler"] 

Writing Dockerfile


### Write ```requirements.txt```

In [4]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0
boto3==1.24.59

pandas==1.2.4

Writing requirements.txt


### Write ```lambda_function.py```

In [5]:
%%writefile lambda_function.py

import pandas as pd
import boto3

# lambda handler
def lambda_handler(event, context):
    # constants
    str_project = '20231010-gen-xii'
    str_model = '01_ad'
    
    # load output from tuning
    print('Loading output from tuning...')
    str_filename = 'df_tuning.csv'
    str_uri = f's3://{str_project}/{str_model}/02_model/02_model/03_lambda_concat_tuning/{str_filename}'
    df = pd.read_csv(str_uri)
    # get iteration
    int_best_iteration = df['iteration'].iloc[0]
    print(f'Best model was from iteration {int_best_iteration}')
    
    # copy model
    print('Copying model...')
    # init
    cls_client = boto3.client('s3')
    # get args
    str_filename = f'dict_model_inference_{int_best_iteration}.pkl'
    str_key_source = f'{str_model}/02_model/02_model/02_batch_tuning/models/{str_filename}'
    dict_copy_source = {
        'Bucket': str_project,
        'Key': str_key_source,
    }
    str_filename_destination = 'final_model.pkl'
    str_key_destination = f'{str_model}/02_model/03_final_model/{str_filename_destination}'
    cls_client.copy_object(
        CopySource=dict_copy_source,
        Bucket=str_project,
        Key=str_key_destination,
    )

Writing lambda_function.py


### Build image and push to ECR

In [6]:
%%sh

# name the image
image=genxii-ad-select-best-model

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon  35.33kB
Step 1/6 : FROM public.ecr.aws/lambda/python:3.8
 ---> 3cd81ffec4d9
Step 2/6 : RUN pip install --upgrade pip
 ---> Using cache
 ---> 1fa305cd0a48
Step 3/6 : COPY requirements.txt  .
 ---> Using cache
 ---> 1d24dd23b2ed
Step 4/6 : RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"
 ---> Using cache
 ---> 2456d2c2e42c
Step 5/6 : COPY lambda_function.py ${LAMBDA_TASK_ROOT}
 ---> Using cache
 ---> 627e0bb0c7ba
Step 6/6 : CMD ["lambda_function.lambda_handler"]
 ---> Using cache
 ---> 3d10d62be6cb
Successfully built 3d10d62be6cb
Successfully tagged genxii-ad-select-best-model:latest


WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'genxii-ad-select-best-model' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-ad-select-best-model]
21182935e60b: Preparing
d985b795e48c: Preparing
494c85511899: Preparing
3f97a2d36016: Preparing
e92756f7b561: Preparing
4fe51bf0bf5c: Preparing
fbbd8c1e2ec1: Preparing
fe2359fe88f2: Preparing
e703f2e518cc: Preparing
97a787951169: Preparing
fbbd8c1e2ec1: Waiting
fe2359fe88f2: Waiting
e703f2e518cc: Waiting
97a787951169: Waiting
4fe51bf0bf5c: Waiting
21182935e60b: Layer already exists
494c85511899: Layer already exists
d985b795e48c: Layer already exists
e92756f7b561: Layer already exists
3f97a2d36016: Layer already exists
4fe51bf0bf5c: Layer already exists
fbbd8c1e2ec1: Layer already exists
e703f2e518cc: Layer already exists
97a787951169: Layer already exists
fe2359fe88f2: Layer already exists
latest: digest: sha256:7edfc8f49559c5929723ea8add786f06fc42329db280d3d0aabd1bff2ab49749 size: 2419


### 2. Create lambda function from image

In [7]:
# initialize class
cls_client_lambda = boto3.client('lambda')

In [8]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

Role: arn:aws:iam::836690756591:role/risk-ops-role


In [9]:
# delete it if it exists
try:
    dict_response = cls_client_lambda.delete_function(
        FunctionName=str_function_name,
    )
    pprint(dict_response)
except:
    pass

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-type': 'application/json',
                                      'date': 'Mon, 22 Apr 2024 19:49:39 GMT',
                                      'x-amzn-requestid': '4c9c789e-31d0-4a45-88ce-4304e89d24f6'},
                      'HTTPStatusCode': 204,
                      'RequestId': '4c9c789e-31d0-4a45-88ce-4304e89d24f6',
                      'RetryAttempts': 0}}


In [10]:
# create function
str_image_uri = f'836690756591.dkr.ecr.us-west-2.amazonaws.com/{str_function_name}:latest' # this must match what we name the image above
dict_response = cls_client_lambda.create_function(
    FunctionName=str_function_name,
    Role=str_role,
    Code={
        'ImageUri': str_image_uri,
    },
    Timeout=60,
    MemorySize=512,
    Publish=True,
    PackageType='Image',
    Architectures=[
        'x86_64',
    ],
    EphemeralStorage={
        'Size': 512,
    },
)
pprint(dict_response)
time.sleep(40)

{'Architectures': ['x86_64'],
 'CodeSha256': '7edfc8f49559c5929723ea8add786f06fc42329db280d3d0aabd1bff2ab49749',
 'CodeSize': 0,
 'Description': '',
 'EphemeralStorage': {'Size': 512},
 'FunctionArn': 'arn:aws:lambda:us-west-2:836690756591:function:genxii-ad-select-best-model',
 'FunctionName': 'genxii-ad-select-best-model',
 'LastModified': '2024-04-22T19:49:39.581+0000',
 'LoggingConfig': {'LogFormat': 'Text',
                   'LogGroup': '/aws/lambda/genxii-ad-select-best-model'},
 'MemorySize': 512,
 'PackageType': 'Image',
 'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '1213',
                                      'content-type': 'application/json',
                                      'date': 'Mon, 22 Apr 2024 19:49:40 GMT',
                                      'x-amzn-requestid': '9ea8b742-d1df-452b-bf6f-c29a0d5b62c2'},
                      'HTTPStatusCode': 201,
                      'RequestId': '9

### Clean-up

In [11]:
for str_file in ['Dockerfile', 'lambda_function.py', 'requirements.txt']:
    os.remove(str_file)